# Parquet - Schema Mismatch

Rename, remove, or change types of expected fields; an external schema contract is required to call the result erroneous.

**Instructions:** Run `prepare_data_parquet.ipynb` to create clean `data/weather.parquet` first (`uv add pyarrow`). This notebook overwrites the Parquet file; restore the clean artifact before rerunning. For combination, use only the required config/function/call, and load/save once.

In [ ]:
from pathlib import Path
from random import Random

import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

# Configuration
DATA_PATH = Path("../../data/weather.parquet")
TIME_COLUMN = "time"
TIMEZONE = "America/Edmonton"
START_DATE = "2025-08-01"
END_DATE = "2025-09-01"  # Exclusive; local dates.
SEED = 814205

# Schema Mismatch
SCHEMA_CANDIDATES = ["temperature_2m", "relative_humidity_2m", "precipitation"]
SCHEMA_KIND = "type_changed"  # renamed, removed, type_changed, mixed, random
SCHEMA_N_FIELDS = 1
RENAME_SUFFIX = "_source"

In [ ]:
def add_schema_mismatch(
    df, rng, *, candidate_columns, kind, n_fields, rename_suffix,
):
    # Parquet has a physical schema, but mismatch requires an EXTERNAL contract.
    if not candidate_columns or len(candidate_columns) != len(set(candidate_columns)):
        raise ValueError("Provide distinct expected candidate columns.")
    if any(c not in df for c in candidate_columns):
        raise ValueError("A candidate field is already absent; start from clean data.")
    if kind not in ("renamed", "removed", "type_changed", "mixed", "random"):
        raise ValueError("Unknown schema change kind.")
    if not isinstance(n_fields, int) or not 1 <= n_fields <= len(candidate_columns):
        raise ValueError("Invalid number of fields.")
    if not rename_suffix:
        raise ValueError("Rename suffix cannot be blank.")
    kinds = ("renamed", "removed", "type_changed")
    selected_kind = rng.choice(kinds + (("mixed",) if n_fields >= 2 else ())) if kind == "random" else kind
    if selected_kind == "mixed" and n_fields < 2:
        raise ValueError("Mixed schema mismatch needs two or more fields.")
    chosen = rng.sample(candidate_columns, k=n_fields)
    if selected_kind == "mixed":
        changes = list(kinds[:min(n_fields, len(kinds))])
        changes += [rng.choice(kinds) for _ in range(n_fields - len(changes))]
        rng.shuffle(changes)
    else:
        changes = [selected_kind] * n_fields
    rename = {c: c + rename_suffix for c, change in zip(chosen, changes) if change == "renamed"}
    removed = [c for c, change in zip(chosen, changes) if change == "removed"]
    recast = [c for c, change in zip(chosen, changes) if change == "type_changed"]
    if len(removed) == len(df.columns):
        raise ValueError("Cannot remove every field; row count would become ambiguous.")
    if any(new in df.columns for new in rename.values()) or len(set(rename.values())) != len(rename):
        raise ValueError("Rename collides with an existing field.")
    if any(not pd.api.types.is_numeric_dtype(df[c]) for c in recast):
        raise ValueError("Type-change candidates must be numeric measurements in this version.")
    dirty = df.copy().rename(columns=rename).drop(columns=removed)
    for c in recast:
        dirty[c] = dirty[c].astype("string")  # Parquet writes a STRING field instead of a numeric field.
    return dirty

In [ ]:
# Small controlled trial (no source file changed).
trial_times = pd.date_range("2025-08-01", periods=7*24, freq="h", tz=TIMEZONE).tz_convert("UTC")
trial = pd.DataFrame({
    "time": trial_times,
    "temperature_2m": [float(10 + i % 17) for i in range(len(trial_times))],
    "relative_humidity_2m": pd.Series([int(40 + i % 40) for i in range(len(trial_times))], dtype="int64"),
    "precipitation": [float(i % 10) for i in range(len(trial_times))],
})

trial_dirty = add_schema_mismatch(trial, Random(SEED),
    candidate_columns=["temperature_2m", "relative_humidity_2m", "precipitation"],
    kind="type_changed", n_fields=1, rename_suffix="_source")
changed = [c for c in trial.columns if c in trial_dirty.columns and str(trial[c].dtype) != str(trial_dirty[c].dtype)]
assert len(changed) == 1
assert len(trial_dirty) == len(trial)
trial_dirty.dtypes

In [ ]:
# Verify actual Parquet types and values on a small temporary file.
from tempfile import TemporaryDirectory
with TemporaryDirectory() as temporary:
    clean_path = Path(temporary) / "clean.parquet"
    temp_path = Path(temporary) / "trial.parquet"
    trial.to_parquet(clean_path, index=False)
    trial_dirty.to_parquet(temp_path, index=False)
    roundtrip = pd.read_parquet(temp_path)
    before_schema = pq.read_schema(clean_path)
    after_schema = pq.read_schema(temp_path)
    assert len(roundtrip) == len(trial_dirty)
    assert roundtrip.columns.tolist() == trial_dirty.columns.tolist()
    assert pq.read_metadata(temp_path).num_rows == len(trial_dirty)
    assert before_schema != after_schema
    print("Stored Parquet schema:", after_schema)

In [ ]:
# Apply the reusable function to a freshly prepared Parquet artifact.
clean = pd.read_parquet(DATA_PATH)
rng = Random(SEED)
dirty = add_schema_mismatch(
    clean, rng, candidate_columns=SCHEMA_CANDIDATES,
    kind=SCHEMA_KIND, n_fields=SCHEMA_N_FIELDS,
    rename_suffix=RENAME_SUFFIX,
)
dirty.to_parquet(DATA_PATH, index=False)
saved = pd.read_parquet(DATA_PATH)

In [ ]:
# Inspect: schema, expected field names and physical Parquet types.
print(saved.dtypes)
print(pq.read_schema(DATA_PATH))

In [ ]:
# Verify persistence: all changes exist after writing the Parquet file.
assert len(saved) == len(dirty)
assert saved.columns.tolist() == dirty.columns.tolist()
assert pq.read_metadata(DATA_PATH).num_rows == len(saved)
assert saved.columns.tolist() != clean.columns.tolist() or any(
    str(saved[c].dtype) != str(clean[c].dtype) for c in saved.columns if c in clean)
print("Physical schema changed relative to the expected input:")
print(pq.read_schema(DATA_PATH))

In [ ]:
# Schema comparisons are clearer as a table than a plot.
pd.DataFrame({"expected": clean.dtypes.astype(str),
              "actual": saved.dtypes.astype(str)}).fillna("<absent>")